In [1]:
#clear all
%reset -f

#import packages
import numpy as np
import scipy
import sys
import os
import pandas as pd
import mne
import matplotlib
import h5py
from sklearn.utils import resample
from mne_icalabel import label_components

root = 'F:/Documents/Science/MirRevAdaptEEG'
participants = list(range(0,32))
#specify which erp we are analyzing
erps = 'frn'
roi = 'medfro'

#pop up plots as separate window & interactive
%matplotlib qt
matplotlib.pyplot.close('all')

In [2]:
#setting up path/ directory
#access trial type epoched data for each participant
def load_tfr_epochs(pp_num, root_dir, erp_path, task):

    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    filename = os.path.join(pp_directory, 'p%03d_%s-epo.fif' % (pp_num, task))

    epochs = mne.read_epochs(filename)

    return epochs, pp_directory

In [3]:
#set up parameters for time-frequency morlet convolution
def set_tfr_params(l_freq = 6, h_freq = 35, num_freq = 50, cycle = 6, samp_rate = 200):
    freqs = np.logspace(*np.log10([l_freq, h_freq]), num = num_freq)
    n_cycles = cycle
    sfreq = samp_rate
    
    return freqs, n_cycles, sfreq

In [4]:
#save tfr'd data
def save_tfr_data(pp_num, root_dir, erp_path, data, task, output):
    # Save the tfr'd data
    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    out_fname = os.path.join(pp_directory, 'p%03d_%s_%s-tfr.h5' % (pp_num, task, output))
    mne.time_frequency.write_tfrs(out_fname, tfr=data, overwrite = True)

In [5]:
#load tfr'd data
def load_tfr_data(pp_num, root_dir, erp_path, task, output):
    # Save the tfr'd data
    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    out_fname = os.path.join(pp_directory, 'p%03d_%s_%s-tfr.h5' % (pp_num, task, output))
    dat = mne.time_frequency.read_tfrs(out_fname)
    
    return dat

In [6]:
# Comparing males and females would be an independent samples comparison
# permutation function should be modified
# F dist used here, but oneway F test = independent t test when dealing with two independent conditions only
def get_ind_clust_perm_test(conditionA, conditionB, pval, n_permutations, n_conditions = 2):
    #define cluster forming threshold based on p-value
    dfn = n_conditions - 1  # degrees of freedom numerator
    dfd = len(participants) - n_conditions  # degrees of freedom denominator
    thresh = scipy.stats.f.ppf(1 - pval, dfn=dfn, dfd=dfd)  
    #run cluster-based permutation test
    F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh, 
                                                          n_permutations = n_permutations, tail = 0, 
                                                          adjacency = None, seed = 999, 
                                                          out_type = 'mask', verbose = True)

    return F_0, clust_idx, clust_pvals, H0

In [7]:
# load in tfr object, ensure to baseline correct; (tfr shape is channels, freqs, timepts)
# narrow down to channels 
# narrow down to frequencies and timepts
# take mean of theta, alpha, or beta ranges (freqs)
# take mean across electrodes (channels)

perturb_conds = ['small_large_aligned', 'smlerrors_rot', 'lrgerrors_rot', 'smlerrors_mir', 'lrgerrors_mir', 'smlerrors_rdm', 'lrgerrors_rdm']

# medial frontal CHANNELS
channels = ['F1', 'Fz', 'F2',
            'FC1', 'FCz', 'FC2',
            'C1', 'Cz', 'C2']


# BASELINE FOR FEEDBACK ONSET
baseline_t = (-0.1, 0)

# THETA
freq_lower = 6
freq_upper = 8

theta_medfro_SmallLargeAligned = []
theta_medfro_SmallRot = []
theta_medfro_LargeRot = []
theta_medfro_SmallMir = []
theta_medfro_LargeMir = []
theta_medfro_SmallRdm = []
theta_medfro_LargeRdm = []
full_theta_medfro_SmallLargeAligned = []
full_theta_medfro_SmallRot = []
full_theta_medfro_LargeRot = []
full_theta_medfro_SmallMir = []
full_theta_medfro_LargeMir = []
full_theta_medfro_SmallRdm = []
full_theta_medfro_LargeRdm = []

for pcond in range(0, len(perturb_conds)):
    medfroppdat = []
    full_medfroppdat = []
    for pp in participants:
        dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = perturb_conds[pcond], output = "power")
        dat = dat[0]*1e12 #convert V^2 to uV^2
        # apply baseline correction
        dat = dat.apply_baseline(baseline_t, mode='mean')
        # narrow down to channels
        dat = dat.pick_channels(channels)
        #transform to ndarray (channels, freqs, times)
        npdat = dat.data

        # get needed frequency indices
        nfreqs = [] #get indices of frequencies we want
        for i in range(0, len(dat.freqs)):
            if dat.freqs[i] >= freq_lower and dat.freqs[i] <= freq_upper:
                nfreqs.append(i)
        npdat = npdat[:, nfreqs, :]
        full_npdat = npdat #all timepts included

        # get needed timept indices
        ntimes = list(range(400,601))
        npdat = npdat[:,:,ntimes]
        
        ppdat = np.mean(npdat, axis = 1) #calcuLarge mean across frequencies
        ppdat = np.mean(ppdat, axis = 0) # calcuLarge mean across channels
        
        full_ppdat = np.mean(full_npdat, axis = 1)
        full_ppdat = np.mean(full_ppdat, axis = 0)
        
        medfroppdat.append(ppdat)
        full_medfroppdat.append(full_ppdat)
        
    if pcond == 0:
        theta_medfro_SmallLargeAligned.append(medfroppdat)
        full_theta_medfro_SmallLargeAligned.append(full_medfroppdat)
    elif pcond == 1:
        theta_medfro_SmallRot.append(medfroppdat)
        full_theta_medfro_SmallRot.append(full_medfroppdat)
    elif pcond == 2:
        theta_medfro_LargeRot.append(medfroppdat)
        full_theta_medfro_LargeRot.append(full_medfroppdat)
    elif pcond == 3:
        theta_medfro_SmallMir.append(medfroppdat)
        full_theta_medfro_SmallMir.append(full_medfroppdat)
    elif pcond == 4:
        theta_medfro_LargeMir.append(medfroppdat)
        full_theta_medfro_LargeMir.append(full_medfroppdat)
    elif pcond == 5:
        theta_medfro_SmallRdm.append(medfroppdat)
        full_theta_medfro_SmallRdm.append(full_medfroppdat)
    elif pcond == 6:
        theta_medfro_LargeRdm.append(medfroppdat)
        full_theta_medfro_LargeRdm.append(full_medfroppdat)

# ALPHA
freq_lower = 9
freq_upper = 13

alpha_medfro_SmallLargeAligned = []
alpha_medfro_SmallRot = []
alpha_medfro_LargeRot = []
alpha_medfro_SmallMir = []
alpha_medfro_LargeMir = []
alpha_medfro_SmallRdm = []
alpha_medfro_LargeRdm = []
full_alpha_medfro_SmallLargeAligned = []
full_alpha_medfro_SmallRot = []
full_alpha_medfro_LargeRot = []
full_alpha_medfro_SmallMir = []
full_alpha_medfro_LargeMir = []
full_alpha_medfro_SmallRdm = []
full_alpha_medfro_LargeRdm = []

for pcond in range(0, len(perturb_conds)):
    medfroppdat = []
    full_medfroppdat = []
    for pp in participants:
        dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = perturb_conds[pcond], output = "power")
        dat = dat[0]*1e12 #convert V^2 to uV^2
        # apply baseline correction
        dat = dat.apply_baseline(baseline_t, mode='mean')
        # narrow down to channels
        dat = dat.pick_channels(channels)
        #transform to ndarray (channels, freqs, times)
        npdat = dat.data

        # get needed frequency indices
        nfreqs = [] #get indices of frequencies we want
        for i in range(0, len(dat.freqs)):
            if dat.freqs[i] >= freq_lower and dat.freqs[i] <= freq_upper:
                nfreqs.append(i)
        npdat = npdat[:, nfreqs, :]
        full_npdat = npdat #all timepts included

        # get needed timept indices
        ntimes = list(range(400,601))
        npdat = npdat[:,:,ntimes]
        
        ppdat = np.mean(npdat, axis = 1) #calcuLarge mean across frequencies
        ppdat = np.mean(ppdat, axis = 0) # calcuLarge mean across channels
        
        full_ppdat = np.mean(full_npdat, axis = 1)
        full_ppdat = np.mean(full_ppdat, axis = 0)
        
        medfroppdat.append(ppdat)
        full_medfroppdat.append(full_ppdat)
        
    if pcond == 0:
        alpha_medfro_SmallLargeAligned.append(medfroppdat)
        full_alpha_medfro_SmallLargeAligned.append(full_medfroppdat)
    elif pcond == 1:
        alpha_medfro_SmallRot.append(medfroppdat)
        full_alpha_medfro_SmallRot.append(full_medfroppdat)
    elif pcond == 2:
        alpha_medfro_LargeRot.append(medfroppdat)
        full_alpha_medfro_LargeRot.append(full_medfroppdat)
    elif pcond == 3:
        alpha_medfro_SmallMir.append(medfroppdat)
        full_alpha_medfro_SmallMir.append(full_medfroppdat)
    elif pcond == 4:
        alpha_medfro_LargeMir.append(medfroppdat)
        full_alpha_medfro_LargeMir.append(full_medfroppdat)
    elif pcond == 5:
        alpha_medfro_SmallRdm.append(medfroppdat)
        full_alpha_medfro_SmallRdm.append(full_medfroppdat)
    elif pcond == 6:
        alpha_medfro_LargeRdm.append(medfroppdat)
        full_alpha_medfro_LargeRdm.append(full_medfroppdat)

# BETA
freq_lower = 13
freq_upper = 25

beta_medfro_SmallLargeAligned = []
beta_medfro_SmallRot = []
beta_medfro_LargeRot = []
beta_medfro_SmallMir = []
beta_medfro_LargeMir = []
beta_medfro_SmallRdm = []
beta_medfro_LargeRdm = []
full_beta_medfro_SmallLargeAligned = []
full_beta_medfro_SmallRot = []
full_beta_medfro_LargeRot = []
full_beta_medfro_SmallMir = []
full_beta_medfro_LargeMir = []
full_beta_medfro_SmallRdm = []
full_beta_medfro_LargeRdm = []

for pcond in range(0, len(perturb_conds)):
    medfroppdat = []
    full_medfroppdat = []
    for pp in participants:
        dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = perturb_conds[pcond], output = "power")
        dat = dat[0]*1e12 #convert V^2 to uV^2
        # apply baseline correction
        dat = dat.apply_baseline(baseline_t, mode='mean')
        # narrow down to channels
        dat = dat.pick_channels(channels)
        #transform to ndarray (channels, freqs, times)
        npdat = dat.data

        # get needed frequency indices
        nfreqs = [] #get indices of frequencies we want
        for i in range(0, len(dat.freqs)):
            if dat.freqs[i] >= freq_lower and dat.freqs[i] <= freq_upper:
                nfreqs.append(i)
        npdat = npdat[:, nfreqs, :]
        full_npdat = npdat #all timepts included

        # get needed timept indices
        ntimes = list(range(400,601))
        npdat = npdat[:,:,ntimes]
        
        ppdat = np.mean(npdat, axis = 1) #calcuLarge mean across frequencies
        ppdat = np.mean(ppdat, axis = 0) # calcuLarge mean across channels
        
        full_ppdat = np.mean(full_npdat, axis = 1)
        full_ppdat = np.mean(full_ppdat, axis = 0)
        
        medfroppdat.append(ppdat)
        full_medfroppdat.append(full_ppdat)
        
    if pcond == 0:
        beta_medfro_SmallLargeAligned.append(medfroppdat)
        full_beta_medfro_SmallLargeAligned.append(full_medfroppdat)
    elif pcond == 1:
        beta_medfro_SmallRot.append(medfroppdat)
        full_beta_medfro_SmallRot.append(full_medfroppdat)
    elif pcond == 2:
        beta_medfro_LargeRot.append(medfroppdat)
        full_beta_medfro_LargeRot.append(full_medfroppdat)
    elif pcond == 3:
        beta_medfro_SmallMir.append(medfroppdat)
        full_beta_medfro_SmallMir.append(full_medfroppdat)
    elif pcond == 4:
        beta_medfro_LargeMir.append(medfroppdat)
        full_beta_medfro_LargeMir.append(full_medfroppdat)
    elif pcond == 5:
        beta_medfro_SmallRdm.append(medfroppdat)
        full_beta_medfro_SmallRdm.append(full_medfroppdat)
    elif pcond == 6:
        beta_medfro_LargeRdm.append(medfroppdat)
        full_beta_medfro_LargeRdm.append(full_medfroppdat)
    

Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/frn\p000_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p001/frn\p001_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p002/frn\p002_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p003/frn\p003_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p004/frn\p004_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p005/frn\p005_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p006/frn\p006_small_large_aligned_power-tfr.h5 ...
Applying baseline co

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p026/frn\p026_smlerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p027/frn\p027_smlerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p028/frn\p028_smlerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p029/frn\p029_smlerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p030/frn\p030_smlerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p031/frn\p031_smlerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/frn\p000_lrgerrors_rot_power-tfr.h5 ...
Applying baseline co

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p021/frn\p021_smlerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p022/frn\p022_smlerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p023/frn\p023_smlerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p024/frn\p024_smlerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p025/frn\p025_smlerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p026/frn\p026_smlerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p027/frn\p027_smlerrors_mir_power-tfr.h5 ...
Applying baseline co

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p016/frn\p016_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p017/frn\p017_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p018/frn\p018_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p019/frn\p019_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p020/frn\p020_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p021/frn\p021_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p022/frn\p022_smlerrors_rdm_power-tfr.h5 ...
Applying baseline co

Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p010/frn\p010_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p011/frn\p011_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p012/frn\p012_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p013/frn\p013_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p014/frn\p014_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p015/frn\p015_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p016/frn\p016_small_large_aligned_power-tfr.h5 ...
Applying baseline co

...

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p026/frn\p026_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p027/frn\p027_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p028/frn\p028_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p029/frn\p029_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p030/frn\p030_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p031/frn\p031_smlerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/frn\p000_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline co

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p020/frn\p020_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p021/frn\p021_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p022/frn\p022_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p023/frn\p023_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p024/frn\p024_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p025/frn\p025_small_large_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p026/frn\p026_small_large_ali

Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p014/frn\p014_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p015/frn\p015_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p016/frn\p016_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p017/frn\p017_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p018/frn\p018_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p019/frn\p019_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p020/frn\p020_lrgerrors_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p009/frn\p009_lrgerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p010/frn\p010_lrgerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p011/frn\p011_lrgerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p012/frn\p012_lrgerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p013/frn\p013_lrgerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p014/frn\p014_lrgerrors_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p015/frn\p015_lrgerrors_mir_power-tfr.h5 ...
Applying baseline co

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p004/frn\p004_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p005/frn\p005_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p006/frn\p006_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p007/frn\p007_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p008/frn\p008_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p009/frn\p009_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p010/frn\p010_lrgerrors_rdm_power-tfr.h5 ...
Applying baseline co

In [8]:
# Get timepts to use for indices given by cluster-based permutation later and for timepts in saved data frames
# But only grab 0 to 1 sec time-locked to feedback onset (idx= 400:601)
root_directory = root
pp = 0 #only need one participant

# we can use aligned data
dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = 'small_large_aligned', output = "power")
full_time = dat[0].times
time = full_time[400:601] #get only timepoints we want

Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/frn\p000_small_large_aligned_power-tfr.h5 ...


In [9]:
# Next, we subtract aligned from each condition, so that we can compare small vs large in each perturbation type

freq_bands = ['theta', 'alpha', 'beta']
diffconds = ['smallrot', 'largerot', 'smallrdm', 'largerdm', 'smallmir', 'largemir']

theta_medfro_smallrot_diff = []
theta_medfro_largerot_diff = []
theta_medfro_smallrdm_diff = []
theta_medfro_largerdm_diff = []
theta_medfro_smallmir_diff = []
theta_medfro_largemir_diff = []
full_theta_medfro_smallrot_diff = []
full_theta_medfro_largerot_diff = []
full_theta_medfro_smallrdm_diff = []
full_theta_medfro_largerdm_diff = []
full_theta_medfro_smallmir_diff = []
full_theta_medfro_largemir_diff = []

alpha_medfro_smallrot_diff = []
alpha_medfro_largerot_diff = []
alpha_medfro_smallrdm_diff = []
alpha_medfro_largerdm_diff = []
alpha_medfro_smallmir_diff = []
alpha_medfro_largemir_diff = []
full_alpha_medfro_smallrot_diff = []
full_alpha_medfro_largerot_diff = []
full_alpha_medfro_smallrdm_diff = []
full_alpha_medfro_largerdm_diff = []
full_alpha_medfro_smallmir_diff = []
full_alpha_medfro_largemir_diff = []

beta_medfro_smallrot_diff = []
beta_medfro_largerot_diff = []
beta_medfro_smallrdm_diff = []
beta_medfro_largerdm_diff = []
beta_medfro_smallmir_diff = []
beta_medfro_largemir_diff = []
full_beta_medfro_smallrot_diff = []
full_beta_medfro_largerot_diff = []
full_beta_medfro_smallrdm_diff = []
full_beta_medfro_largerdm_diff = []
full_beta_medfro_smallmir_diff = []
full_beta_medfro_largemir_diff = []

for band in range(0, len(freq_bands)):
    if band == 0:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(theta_medfro_SmallRot[0], theta_medfro_SmallLargeAligned[0])
                theta_medfro_smallrot_diff.append(diffevks)
                theta_medfro_smallrot_diff = theta_medfro_smallrot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_theta_medfro_SmallRot[0], full_theta_medfro_SmallLargeAligned[0])
                full_theta_medfro_smallrot_diff.append(full_diffevks)
                full_theta_medfro_smallrot_diff = full_theta_medfro_smallrot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(theta_medfro_LargeRot[0], theta_medfro_SmallLargeAligned[0])
                theta_medfro_largerot_diff.append(diffevks)
                theta_medfro_largerot_diff = theta_medfro_largerot_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_LargeRot[0], full_theta_medfro_SmallLargeAligned[0])
                full_theta_medfro_largerot_diff.append(full_diffevks)
                full_theta_medfro_largerot_diff = full_theta_medfro_largerot_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(theta_medfro_SmallRdm[0], theta_medfro_SmallLargeAligned[0])
                theta_medfro_smallrdm_diff.append(diffevks)
                theta_medfro_smallrdm_diff = theta_medfro_smallrdm_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_SmallRdm[0], full_theta_medfro_SmallLargeAligned[0])
                full_theta_medfro_smallrdm_diff.append(full_diffevks)
                full_theta_medfro_smallrdm_diff = full_theta_medfro_smallrdm_diff[0]
                
            elif cond == 3:
                diffevks = np.subtract(theta_medfro_LargeRdm[0], theta_medfro_SmallLargeAligned[0])
                theta_medfro_largerdm_diff.append(diffevks)
                theta_medfro_largerdm_diff = theta_medfro_largerdm_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_LargeRdm[0], full_theta_medfro_SmallLargeAligned[0])
                full_theta_medfro_largerdm_diff.append(full_diffevks)
                full_theta_medfro_largerdm_diff = full_theta_medfro_largerdm_diff[0]
                
            elif cond == 4:
                diffevks = np.subtract(theta_medfro_SmallMir[0], theta_medfro_SmallLargeAligned[0])
                theta_medfro_smallmir_diff.append(diffevks)
                theta_medfro_smallmir_diff = theta_medfro_smallmir_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_SmallMir[0], full_theta_medfro_SmallLargeAligned[0])
                full_theta_medfro_smallmir_diff.append(full_diffevks)
                full_theta_medfro_smallmir_diff = full_theta_medfro_smallmir_diff[0]
                
            elif cond == 5:
                diffevks = np.subtract(theta_medfro_LargeMir[0], theta_medfro_SmallLargeAligned[0])
                theta_medfro_largemir_diff.append(diffevks)
                theta_medfro_largemir_diff = theta_medfro_largemir_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_LargeMir[0], full_theta_medfro_SmallLargeAligned[0])
                full_theta_medfro_largemir_diff.append(full_diffevks)
                full_theta_medfro_largemir_diff = full_theta_medfro_largemir_diff[0]
                
    elif band == 1:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(alpha_medfro_SmallRot[0], alpha_medfro_SmallLargeAligned[0])
                alpha_medfro_smallrot_diff.append(diffevks)
                alpha_medfro_smallrot_diff = alpha_medfro_smallrot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_alpha_medfro_SmallRot[0], full_alpha_medfro_SmallLargeAligned[0])
                full_alpha_medfro_smallrot_diff.append(full_diffevks)
                full_alpha_medfro_smallrot_diff = full_alpha_medfro_smallrot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(alpha_medfro_LargeRot[0], alpha_medfro_SmallLargeAligned[0])
                alpha_medfro_largerot_diff.append(diffevks)
                alpha_medfro_largerot_diff = alpha_medfro_largerot_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_LargeRot[0], full_alpha_medfro_SmallLargeAligned[0])
                full_alpha_medfro_largerot_diff.append(full_diffevks)
                full_alpha_medfro_largerot_diff = full_alpha_medfro_largerot_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(alpha_medfro_SmallRdm[0], alpha_medfro_SmallLargeAligned[0])
                alpha_medfro_smallrdm_diff.append(diffevks)
                alpha_medfro_smallrdm_diff = alpha_medfro_smallrdm_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_SmallRdm[0], full_alpha_medfro_SmallLargeAligned[0])
                full_alpha_medfro_smallrdm_diff.append(full_diffevks)
                full_alpha_medfro_smallrdm_diff = full_alpha_medfro_smallrdm_diff[0]
                
            elif cond == 3:
                diffevks = np.subtract(alpha_medfro_LargeRdm[0], alpha_medfro_SmallLargeAligned[0])
                alpha_medfro_largerdm_diff.append(diffevks)
                alpha_medfro_largerdm_diff = alpha_medfro_largerdm_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_LargeRdm[0], full_alpha_medfro_SmallLargeAligned[0])
                full_alpha_medfro_largerdm_diff.append(full_diffevks)
                full_alpha_medfro_largerdm_diff = full_alpha_medfro_largerdm_diff[0]
                
            elif cond == 4:
                diffevks = np.subtract(alpha_medfro_SmallMir[0], alpha_medfro_SmallLargeAligned[0])
                alpha_medfro_smallmir_diff.append(diffevks)
                alpha_medfro_smallmir_diff = alpha_medfro_smallmir_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_SmallMir[0], full_alpha_medfro_SmallLargeAligned[0])
                full_alpha_medfro_smallmir_diff.append(full_diffevks)
                full_alpha_medfro_smallmir_diff = full_alpha_medfro_smallmir_diff[0]
                
            elif cond == 5:
                diffevks = np.subtract(alpha_medfro_LargeMir[0], alpha_medfro_SmallLargeAligned[0])
                alpha_medfro_largemir_diff.append(diffevks)
                alpha_medfro_largemir_diff = alpha_medfro_largemir_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_LargeMir[0], full_alpha_medfro_SmallLargeAligned[0])
                full_alpha_medfro_largemir_diff.append(full_diffevks)
                full_alpha_medfro_largemir_diff = full_alpha_medfro_largemir_diff[0]
                
    elif band == 2:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(beta_medfro_SmallRot[0], beta_medfro_SmallLargeAligned[0])
                beta_medfro_smallrot_diff.append(diffevks)
                beta_medfro_smallrot_diff = beta_medfro_smallrot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_beta_medfro_SmallRot[0], full_beta_medfro_SmallLargeAligned[0])
                full_beta_medfro_smallrot_diff.append(full_diffevks)
                full_beta_medfro_smallrot_diff = full_beta_medfro_smallrot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(beta_medfro_LargeRot[0], beta_medfro_SmallLargeAligned[0])
                beta_medfro_largerot_diff.append(diffevks)
                beta_medfro_largerot_diff = beta_medfro_largerot_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_LargeRot[0], full_beta_medfro_SmallLargeAligned[0])
                full_beta_medfro_largerot_diff.append(full_diffevks)
                full_beta_medfro_largerot_diff = full_beta_medfro_largerot_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(beta_medfro_SmallRdm[0], beta_medfro_SmallLargeAligned[0])
                beta_medfro_smallrdm_diff.append(diffevks)
                beta_medfro_smallrdm_diff = beta_medfro_smallrdm_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_SmallRdm[0], full_beta_medfro_SmallLargeAligned[0])
                full_beta_medfro_smallrdm_diff.append(full_diffevks)
                full_beta_medfro_smallrdm_diff = full_beta_medfro_smallrdm_diff[0]
                
            elif cond == 3:
                diffevks = np.subtract(beta_medfro_LargeRdm[0], beta_medfro_SmallLargeAligned[0])
                beta_medfro_largerdm_diff.append(diffevks)
                beta_medfro_largerdm_diff = beta_medfro_largerdm_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_LargeRdm[0], full_beta_medfro_SmallLargeAligned[0])
                full_beta_medfro_largerdm_diff.append(full_diffevks)
                full_beta_medfro_largerdm_diff = full_beta_medfro_largerdm_diff[0]
                
            elif cond == 4:
                diffevks = np.subtract(beta_medfro_SmallMir[0], beta_medfro_SmallLargeAligned[0])
                beta_medfro_smallmir_diff.append(diffevks)
                beta_medfro_smallmir_diff = beta_medfro_smallmir_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_SmallMir[0], full_beta_medfro_SmallLargeAligned[0])
                full_beta_medfro_smallmir_diff.append(full_diffevks)
                full_beta_medfro_smallmir_diff = full_beta_medfro_smallmir_diff[0]
                
            elif cond == 5:
                diffevks = np.subtract(beta_medfro_LargeMir[0], beta_medfro_SmallLargeAligned[0])
                beta_medfro_largemir_diff.append(diffevks)
                beta_medfro_largemir_diff = beta_medfro_largemir_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_LargeMir[0], full_beta_medfro_SmallLargeAligned[0])
                full_beta_medfro_largemir_diff.append(full_diffevks)
                full_beta_medfro_largemir_diff = full_beta_medfro_largemir_diff[0]

In [10]:
# Next step is to subtract small from large condition, to generate a single signal for each perturbation
diffconds = ['rot', 'rdm', 'mir']
freq_bands = ['theta', 'alpha', 'beta']

theta_medfro_rot_diff = []
theta_medfro_rdm_diff = []
theta_medfro_mir_diff = []
full_theta_medfro_rot_diff = []
full_theta_medfro_rdm_diff = []
full_theta_medfro_mir_diff = []

alpha_medfro_rot_diff = []
alpha_medfro_rdm_diff = []
alpha_medfro_mir_diff = []
full_alpha_medfro_rot_diff = []
full_alpha_medfro_rdm_diff = []
full_alpha_medfro_mir_diff = []

beta_medfro_rot_diff = []
beta_medfro_rdm_diff = []
beta_medfro_mir_diff = []
full_beta_medfro_rot_diff = []
full_beta_medfro_rdm_diff = []
full_beta_medfro_mir_diff = []

for band in range(0, len(freq_bands)):
    if band == 0:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(theta_medfro_largerot_diff, theta_medfro_smallrot_diff)
                theta_medfro_rot_diff.append(diffevks)
                theta_medfro_rot_diff = theta_medfro_rot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_theta_medfro_largerot_diff, full_theta_medfro_smallrot_diff)
                full_theta_medfro_rot_diff.append(full_diffevks)
                full_theta_medfro_rot_diff = full_theta_medfro_rot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(theta_medfro_largerdm_diff, theta_medfro_smallrdm_diff)
                theta_medfro_rdm_diff.append(diffevks)
                theta_medfro_rdm_diff = theta_medfro_rdm_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_largerdm_diff, full_theta_medfro_smallrdm_diff)
                full_theta_medfro_rdm_diff.append(full_diffevks)
                full_theta_medfro_rdm_diff = full_theta_medfro_rdm_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(theta_medfro_largemir_diff, theta_medfro_smallmir_diff)
                theta_medfro_mir_diff.append(diffevks)
                theta_medfro_mir_diff = theta_medfro_mir_diff[0]
                
                full_diffevks = np.subtract(full_theta_medfro_largemir_diff, full_theta_medfro_smallmir_diff)
                full_theta_medfro_mir_diff.append(full_diffevks)
                full_theta_medfro_mir_diff = full_theta_medfro_mir_diff[0]
                
    elif band == 1:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(alpha_medfro_largerot_diff, alpha_medfro_smallrot_diff)
                alpha_medfro_rot_diff.append(diffevks)
                alpha_medfro_rot_diff = alpha_medfro_rot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_alpha_medfro_largerot_diff, full_alpha_medfro_smallrot_diff)
                full_alpha_medfro_rot_diff.append(full_diffevks)
                full_alpha_medfro_rot_diff = full_alpha_medfro_rot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(alpha_medfro_largerdm_diff, alpha_medfro_smallrdm_diff)
                alpha_medfro_rdm_diff.append(diffevks)
                alpha_medfro_rdm_diff = alpha_medfro_rdm_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_largerdm_diff, full_alpha_medfro_smallrdm_diff)
                full_alpha_medfro_rdm_diff.append(full_diffevks)
                full_alpha_medfro_rdm_diff = full_alpha_medfro_rdm_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(alpha_medfro_largemir_diff, alpha_medfro_smallmir_diff)
                alpha_medfro_mir_diff.append(diffevks)
                alpha_medfro_mir_diff = alpha_medfro_mir_diff[0]
                
                full_diffevks = np.subtract(full_alpha_medfro_largemir_diff, full_alpha_medfro_smallmir_diff)
                full_alpha_medfro_mir_diff.append(full_diffevks)
                full_alpha_medfro_mir_diff = full_alpha_medfro_mir_diff[0]
                
    elif band == 2:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(beta_medfro_largerot_diff, beta_medfro_smallrot_diff)
                beta_medfro_rot_diff.append(diffevks)
                beta_medfro_rot_diff = beta_medfro_rot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_beta_medfro_largerot_diff, full_beta_medfro_smallrot_diff)
                full_beta_medfro_rot_diff.append(full_diffevks)
                full_beta_medfro_rot_diff = full_beta_medfro_rot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(beta_medfro_largerdm_diff, beta_medfro_smallrdm_diff)
                beta_medfro_rdm_diff.append(diffevks)
                beta_medfro_rdm_diff = beta_medfro_rdm_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_largerdm_diff, full_beta_medfro_smallrdm_diff)
                full_beta_medfro_rdm_diff.append(full_diffevks)
                full_beta_medfro_rdm_diff = full_beta_medfro_rdm_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(beta_medfro_largemir_diff, beta_medfro_smallmir_diff)
                beta_medfro_mir_diff.append(diffevks)
                beta_medfro_mir_diff = beta_medfro_mir_diff[0]
                
                full_diffevks = np.subtract(full_beta_medfro_largemir_diff, full_beta_medfro_smallmir_diff)
                full_beta_medfro_mir_diff.append(full_diffevks)
                full_beta_medfro_mir_diff = full_beta_medfro_mir_diff[0]

In [11]:
#separate by sex

#beta
beta_medfro_rot_males = beta_medfro_rot_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
beta_medfro_rot_females = beta_medfro_rot_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

beta_medfro_rdm_males = beta_medfro_rdm_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
beta_medfro_rdm_females = beta_medfro_rdm_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

beta_medfro_mir_males = beta_medfro_mir_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
beta_medfro_mir_females = beta_medfro_mir_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

#alpha
alpha_medfro_rot_males = alpha_medfro_rot_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
alpha_medfro_rot_females = alpha_medfro_rot_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

alpha_medfro_rdm_males = alpha_medfro_rdm_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
alpha_medfro_rdm_females = alpha_medfro_rdm_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

alpha_medfro_mir_males = alpha_medfro_mir_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
alpha_medfro_mir_females = alpha_medfro_mir_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

#theta
theta_medfro_rot_males = theta_medfro_rot_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
theta_medfro_rot_females = theta_medfro_rot_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

theta_medfro_rdm_males = theta_medfro_rdm_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
theta_medfro_rdm_females = theta_medfro_rdm_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

theta_medfro_mir_males = theta_medfro_mir_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
theta_medfro_mir_females = theta_medfro_mir_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

In [12]:
# Compare SMALL vs LARGE for each perturbation
# Generate a data frame to tabulate condition, cluster indices, cluster timepts, p values
# This information can then be included in plots
p = 0.05
perms = 1000

condition = []
clust_idx_start = []
clust_idx_end = []
time_start = []
time_end = []
p_values = []

freq_bands = ['theta', 'alpha', 'beta']

for band in range(0, len(freq_bands)):
    if band == 0:
        theta_conditionnames = ['theta_medfro_rot_mvf', 'theta_medfro_rdm_mvf', 'theta_medfro_mir_mvf']
        for c in range(0, len(theta_conditionnames)):
            if c == 0:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(theta_medfro_rot_males, theta_medfro_rot_females, p, perms)
        #         print(clust_idx, clust_pvals)
                if len(clust_idx) == 0:
                    condition.append(theta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(theta_conditionnames[c])
            
            elif c == 1:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(theta_medfro_rdm_males, theta_medfro_rdm_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(theta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(theta_conditionnames[c])
            
            elif c == 2:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(theta_medfro_mir_males, theta_medfro_mir_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(theta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(theta_conditionnames[c])
    elif band == 1:
        alpha_conditionnames = ['alpha_medfro_rot_mvf', 'alpha_medfro_rdm_mvf', 'alpha_medfro_mir_mvf']
        for c in range(0, len(alpha_conditionnames)):
            if c == 0:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(alpha_medfro_rot_males, alpha_medfro_rot_females, p, perms)
        #         print(clust_idx, clust_pvals)
                if len(clust_idx) == 0:
                    condition.append(alpha_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(alpha_conditionnames[c])
            
            elif c == 1:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(alpha_medfro_rdm_males, alpha_medfro_rdm_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(alpha_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(alpha_conditionnames[c])
            
            elif c == 2:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(alpha_medfro_mir_males, alpha_medfro_mir_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(alpha_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(alpha_conditionnames[c])
            
    elif band == 2:
        beta_conditionnames = ['beta_medfro_rot_mvf', 'beta_medfro_rdm_mvf', 'beta_medfro_mir_mvf']
        for c in range(0, len(beta_conditionnames)):
            if c == 0:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(beta_medfro_rot_males, beta_medfro_rot_females, p, perms)
        #         print(clust_idx, clust_pvals)
                if len(clust_idx) == 0:
                    condition.append(beta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(beta_conditionnames[c])
            
            elif c == 1:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(beta_medfro_rdm_males, beta_medfro_rdm_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(beta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(beta_conditionnames[c])
            
            elif c == 2:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(beta_medfro_mir_males, beta_medfro_mir_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(beta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(beta_conditionnames[c])
        
perm_test = pd.DataFrame(
    {'condition': condition,
     'clust_idx_start': clust_idx_start,
     'clust_idx_end': clust_idx_end,
     'time_start': time_start,
     'time_end': time_end,
     'p_values': p_values})

perm_test_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/sex_diff/', 'TFR_Permutation_test_SvL_PerturbTypeComp_SexDIFF_%s_%s.csv' % (erps, roi))
perm_test.to_csv(perm_test_filename)

stat_fun(H1): min=0.000424 max=1.413628
Running initial clustering …
Found 0 clusters
stat_fun(H1): min=0.000003 max=2.241528
Running initial clustering …
Found 0 clusters
stat_fun(H1): min=0.000120 max=1.718441
Running initial clustering …
Found 0 clusters
stat_fun(H1): min=0.000000 max=6.246655
Running initial clustering …
Found 1 cluster


C:\Users\Raphael\AppData\Local\Temp\ipykernel_17588\4221181891.py:10: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh,
C:\Users\Raphael\AppData\Local\Temp\ipykernel_17588\4221181891.py:10: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh,
C:\Users\Raphael\AppData\Local\Temp\ipykernel_17588\4221181891.py:10: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh,


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=0.000001 max=1.640763
Running initial clustering …
Found 0 clusters
stat_fun(H1): min=0.244265 max=6.546799
Running initial clustering …
Found 2 clusters


C:\Users\Raphael\AppData\Local\Temp\ipykernel_17588\4221181891.py:10: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh,


...

  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=0.000018 max=7.127784
Running initial clustering …
Found 1 cluster


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=0.000040 max=3.587944
Running initial clustering …
Found 0 clusters


C:\Users\Raphael\AppData\Local\Temp\ipykernel_17588\4221181891.py:10: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh,
